# TechMarket — анализ данных на Python

**Цель:** исследовать продажи интернет-магазина, динамику выручки, ассортимент и повторные покупки клиентов.

**Стек:** Python, pandas, NumPy, Matplotlib, Seaborn, Jupyter.

## Методология и допущения
- Исходный файл: `../data/techmarket_python_case.xlsx` (путь относительно папки `notebooks/`).
- Для метрик продаж учитываются только заказы со статусом `completed`. `cancelled` и `returned` не считаются реализованными продажами.
- Выручка пересчитывается как `quantity × price × (1 − discount)`. Строки без цены нельзя корректно оценить, поэтому они исключаются из расчётов выручки; исходные данные при этом не перезаписываются.
- Пропущенный `customer_id` не заполняется вымышленным идентификатором. Такие строки остаются в анализе продаж, но не участвуют в клиентских метриках.
- Значение скидки `0.8` не исправляется автоматически: оно сохраняется как 80%, поскольку без подтверждения нельзя считать его ошибкой.
- XYZ-анализ строится на месячных продажах с нулевыми значениями в месяцах без продаж. Коэффициент вариации рассчитывается при наличии минимум двух месяцев и ненулевого среднего.


## 1. Импорт библиотек и загрузка данных

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/techmarket_python_case.xlsx")
df = pd.read_excel(DATA_PATH)

print(f"Размер датасета: {df.shape[0]:,} строк × {df.shape[1]} столбцов")
display(df.head())


## 2. Первичное исследование данных

In [ ]:
display(df.info())
display(df.describe(include="all").T)
display(df.isna().sum().rename("missing_values").to_frame())
display(df["status"].value_counts(dropna=False))
display(df[["order_date", "quantity", "price", "discount"]].agg(["min", "max"]))


## 3. Очистка и подготовка данных
Не используем случайные значения для заполнения идентификаторов или цен. Сначала нормализуем категории статусов и проверим дубликаты.

In [ ]:
df_clean = df.drop_duplicates().copy()

df_clean["order_date"] = pd.to_datetime(df_clean["order_date"], errors="coerce")
df_clean["status"] = df_clean["status"].astype("string").str.strip().str.lower()
df_clean["customer_id"] = pd.to_numeric(df_clean["customer_id"], errors="coerce").astype("Int64")

# Проверки значений, которые могут нарушить расчёты
quality_checks = {
    "duplicate_rows_after_deduplication": int(df_clean.duplicated().sum()),
    "missing_customer_id": int(df_clean["customer_id"].isna().sum()),
    "missing_price": int(df_clean["price"].isna().sum()),
    "missing_order_date": int(df_clean["order_date"].isna().sum()),
    "non_positive_quantity": int((df_clean["quantity"] <= 0).sum()),
    "negative_price": int((df_clean["price"] < 0).sum()),
    "discount_outside_0_1": int((~df_clean["discount"].between(0, 1)).sum()),
}
display(pd.Series(quality_checks, name="count").to_frame())


## 4. Формирование аналитической выборки
Для анализа продаж берём только `completed`. Неизвестный клиент не мешает учитывать продажу, но будет исключён из клиентских метрик. Строки без цены или с некорректными ключевыми полями не участвуют в денежном анализе.

In [ ]:
sales = df_clean.loc[
    df_clean["status"].eq("completed")
    & df_clean["order_date"].notna()
    & df_clean["quantity"].gt(0)
    & df_clean["price"].notna()
    & df_clean["price"].ge(0)
    & df_clean["discount"].between(0, 1)
].copy()

sales["month"] = sales["order_date"].dt.to_period("M")
sales["revenue_calc"] = (
    sales["quantity"] * sales["price"] * (1 - sales["discount"])
)

print(f"Строк в исходных данных: {len(df):,}")
print(f"Строк после удаления полных дубликатов: {len(df_clean):,}")
print(f"Строк в аналитической выборке продаж: {len(sales):,}")
print(f"Уникальных заказов в выборке: {sales['order_id'].nunique():,}")
display(sales.head())


### Проверка исходной и пересчитанной выручки
Сравнение помогает понять, соответствует ли готовая колонка `revenue` формуле на основе цены, количества и скидки.

In [ ]:
sales["revenue_difference"] = sales["revenue"] - sales["revenue_calc"]
display(sales[["revenue", "revenue_calc", "revenue_difference"]].describe())
print("Количество строк с отличием более 0.01:",
      int(sales["revenue_difference"].abs().gt(0.01).sum()))


## 5. Динамика продаж

In [ ]:
monthly_metrics = (
    sales.groupby("month")
    .agg(
        revenue=("revenue_calc", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique"),
        quantity=("quantity", "sum"),
    )
    .sort_index()
)
monthly_metrics["avg_order_value"] = (
    monthly_metrics["revenue"] / monthly_metrics["orders"]
)
monthly_metrics["items_per_order"] = (
    monthly_metrics["quantity"] / monthly_metrics["orders"]
)
monthly_metrics["revenue_mom_pct"] = monthly_metrics["revenue"].pct_change() * 100

display(monthly_metrics.round(2))


In [ ]:
monthly_metrics[["revenue", "orders", "avg_order_value"]].plot(
    subplots=True, figsize=(12, 9), marker="o", layout=(3, 1),
    title=["Monthly revenue", "Completed orders", "Average order value"],
    legend=False
)
plt.tight_layout()
plt.show()


## 6. Анализ категорий

In [ ]:
category_revenue = (
    sales.groupby("category")["revenue_calc"]
    .sum()
    .sort_values(ascending=False)
)
display(category_revenue.to_frame("revenue"))
category_month = sales.pivot_table(
    index="month", columns="category", values="revenue_calc",
    aggfunc="sum", fill_value=0
)
display(category_month)
display((category_month.div(category_month.sum(axis=1), axis=0) * 100).round(2))


## 7. ABC-анализ товаров
Классификация по накопительной доле выручки: A — до 80%, B — следующие до 95%, C — остаток. Граница может быть пересечена одной товарной позицией.

In [ ]:
abc = (
    sales.groupby("product", as_index=False)["revenue_calc"]
    .sum()
    .sort_values("revenue_calc", ascending=False)
    .reset_index(drop=True)
)
total_revenue = abc["revenue_calc"].sum()
abc["revenue_share_pct"] = abc["revenue_calc"] / total_revenue * 100
abc["cumulative_share_pct"] = abc["revenue_share_pct"].cumsum()
abc["ABC"] = np.select(
    [abc["cumulative_share_pct"].le(80),
     abc["cumulative_share_pct"].le(95)],
    ["A", "B"],
    default="C"
)
display(abc)
display(abc.groupby("ABC").agg(
    products=("product", "count"),
    revenue=("revenue_calc", "sum")
))


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.bar(abc["product"], abc["revenue_calc"])
ax1.set_ylabel("Revenue")
ax1.tick_params(axis="x", rotation=90)
ax2 = ax1.twinx()
ax2.plot(abc["product"], abc["cumulative_share_pct"], marker="o")
ax2.set_ylabel("Cumulative revenue share, %")
ax2.axhline(80, linestyle="--")
ax2.axhline(95, linestyle=":")
plt.title("ABC / Pareto analysis by product")
plt.tight_layout()
plt.show()


## 8. XYZ-анализ товаров
Для каждого товара формируется полный календарь месяцев в периоде наблюдения; отсутствие продаж в месяце трактуется как 0. CV — стандартное отклонение / среднее × 100. Для среднего, равного нулю, CV не определён.

In [ ]:
all_months = pd.period_range(sales["month"].min(), sales["month"].max(), freq="M")
monthly_product_qty = (
    sales.pivot_table(index="product", columns="month", values="quantity",
                      aggfunc="sum", fill_value=0)
    .reindex(columns=all_months, fill_value=0)
)
xyz = pd.DataFrame(index=monthly_product_qty.index)
xyz["mean_monthly_qty"] = monthly_product_qty.mean(axis=1)
xyz["cv_pct"] = (
    monthly_product_qty.std(axis=1, ddof=0)
    / xyz["mean_monthly_qty"].replace(0, np.nan) * 100
)
xyz["XYZ"] = np.select(
    [xyz["cv_pct"].le(10), xyz["cv_pct"].le(25)],
    ["X", "Y"],
    default="Z"
)
xyz.loc[xyz["cv_pct"].isna(), "XYZ"] = "н/д"
display(xyz.sort_values("cv_pct"))


## 9. Когортный анализ повторных покупок
Когорта определяется месяцем первой завершённой покупки. Метрика — доля клиентов когорты, совершивших покупку в соответствующий месяц. Это когортный retention по месяцам, а не показатель активности внутри продукта.

In [ ]:
customer_orders = sales.dropna(subset=["customer_id"]).copy()
customer_orders["order_month"] = customer_orders["order_date"].dt.to_period("M")
customer_orders = customer_orders.drop_duplicates(["customer_id", "order_id"])

customer_orders["cohort_month"] = (
    customer_orders.groupby("customer_id")["order_month"].transform("min")
)
customer_orders["cohort_index"] = (
    (customer_orders["order_month"].dt.year - customer_orders["cohort_month"].dt.year) * 12
    + customer_orders["order_month"].dt.month
    - customer_orders["cohort_month"].dt.month
)
cohort_counts = customer_orders.groupby(
    ["cohort_month", "cohort_index"]
)["customer_id"].nunique().unstack(fill_value=0)

cohort_sizes = cohort_counts.iloc[:, 0]
retention_pct = cohort_counts.div(cohort_sizes, axis=0) * 100
display(retention_pct.round(1))


In [ ]:
plt.figure(figsize=(12, 7))
sns.heatmap(retention_pct, annot=True, fmt=".0f", cmap="YlGnBu",
            vmin=0, vmax=100, cbar_kws={"label": "Retention, %"})
plt.title("Monthly customer cohort retention")
plt.xlabel("Months since first completed purchase")
plt.ylabel("First purchase cohort")
plt.tight_layout()
plt.show()


## 10. Выводы и дальнейшие шаги
Заполни этот раздел после проверки фактических результатов выполнения ноутбука:

- Какие месяцы показали максимальную выручку и за счёт чего?
- Какие товары формируют основную долю выручки (ABC)?
- Насколько стабилен спрос по товарам (XYZ)?
- Как меняется доля повторных покупок по когортам?
- Какие ограничения данных влияют на интерпретацию?

**Ограничение:** результаты относятся к предоставленному учебному датасету и не являются доказательством реального коммерческого эффекта.